<a href="https://colab.research.google.com/github/dataguirre/curso-ia-ciencia-de-datos/blob/main/workshops/03-workshop-mejora-llms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA para ciencia de datos: Workshop 3

En el Workshop 2 vimos como **usar** un LLM. Ahora vamos a **mejorarlo**, de lo mas barato a lo mas costoso:

1. **Prompting**: instrucciones y ejemplos. Gratis e inmediato.
2. **Contexto**: darle un documento dentro del prompt.
3. **LoRA**: ajustar una fraccion minuscula de los pesos.

Trabajamos con dos de las empresas del curso:

- **Tienda de e-commerce**: necesita categorizar automaticamente los productos de su catalogo.
- **Empresa de transporte**: quiere que un asistente responda preguntas sobre su manual operativo.

> **Antes de empezar:** `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion` y selecciona **GPU (T4)**. La Actividad 3 la necesita.


### Configuracion inicial

Montamos el cliente de Groq y la funcion `preguntar_groq` del Workshop 2.

`reasoning_effort="low"` porque `gpt-oss-20b` es un modelo de razonamiento: gasta tokens "pensando" antes de escribir, y para clasificar no lo necesitamos.


In [ ]:
!pip install -q groq

from groq import Groq
from google.colab import userdata

MODELO_GROQ = "openai/gpt-oss-20b"
client = Groq(api_key=userdata.get("GROQ_API_KEY"))


def preguntar_groq(prompt: str, system_prompt: str = None, modelo: str = MODELO_GROQ, temperature: float = 0.7) -> str:
  """Envia un prompt (y opcionalmente un system prompt) a un LLM de Groq y devuelve el texto de la respuesta."""

  mensajes = []
  if system_prompt:
    mensajes.append({"role": "system", "content": system_prompt})
  mensajes.append({"role": "user", "content": prompt})

  respuesta = client.chat.completions.create(
      model=modelo,
      messages=mensajes,
      temperature=temperature,
      reasoning_effort="low",
  )

  return respuesta.choices[0].message.content


## Actividad 1: Via prompting

### Objetivo

La tienda de e-commerce quiere clasificar cada producto de su catalogo en una categoria. Vamos a resolverlo **sin entrenar nada**, solo con instrucciones y ejemplos.

Implementaremos `preguntar_con_estilo(prompt, system_prompt=None, ejemplos=None)`, que arma el mensaje con **system prompt** + **ejemplos few-shot** + la pregunta final.

`ejemplos` es una lista de tuplas `(entrada, salida_esperada)` que se insertan como turnos `user`/`assistant`, como si el modelo ya hubiera respondido bien antes.

> Guarda esta idea: en la Actividad 3 resolveremos **esta misma tarea** con LoRA, y compararemos.


#### Tarea 1: Construir la funcion `preguntar_con_estilo`

### Requisitos

1. Si hay `system_prompt`, va primero como mensaje `system`.
2. Por cada tupla de `ejemplos`, agregar **dos** mensajes en orden: uno `user` con la entrada y uno `assistant` con la salida.
3. Al final, el mensaje `user` con el `prompt` real.


In [ ]:
def preguntar_con_estilo(prompt: str, system_prompt: str = None, ejemplos: list = None, modelo: str = MODELO_GROQ) -> str:
  """Arma un prompt con system + ejemplos few-shot + pregunta final, y llama a Groq."""

  mensajes = []

  # TODO: si system_prompt no es None, agregarlo como {"role": "system", ...}

  # TODO: recorrer "ejemplos" (tuplas (entrada, salida)) y agregar por cada una
  #       un mensaje {"role": "user", ...} y luego uno {"role": "assistant", ...}
  #       Pista: for entrada, salida in (ejemplos or []):

  # TODO: agregar al final el mensaje de usuario con "prompt"

  respuesta = client.chat.completions.create(
      model=modelo,
      messages=mensajes,
      temperature=0,
      reasoning_effort="low",
  )

  return respuesta.choices[0].message.content


**Evaluacion de implementacion**

In [ ]:
# @title
# Celda de validacion. No modificar.
from unittest.mock import patch, MagicMock

def _respuesta_falsa(texto="ok"):
  resp = MagicMock()
  resp.choices = [MagicMock()]
  resp.choices[0].message.content = texto
  return resp

def _validar_preguntar_con_estilo():
  if "preguntar_con_estilo" not in globals():
    print("\u2717 Todavia no existe 'preguntar_con_estilo'. Ejecuta la celda anterior.")
    return

  fallas = []
  ejemplos = [("hola", "mundo"), ("uno", "dos")]

  with patch.object(client.chat.completions, "create", return_value=_respuesta_falsa()) as mock_create:
    try:
      preguntar_con_estilo("pregunta final", system_prompt="eres breve", ejemplos=ejemplos)
    except Exception as e:
      print(f"\u2717 preguntar_con_estilo(...) lanzo {type(e).__name__}: {e}")
      return

    if mock_create.call_args is None:
      print("\u2717 Nunca se llamo a client.chat.completions.create.")
      return

    mensajes = mock_create.call_args.kwargs.get("messages", [])
    esperado = [
        {"role": "system", "content": "eres breve"},
        {"role": "user", "content": "hola"},
        {"role": "assistant", "content": "mundo"},
        {"role": "user", "content": "uno"},
        {"role": "assistant", "content": "dos"},
        {"role": "user", "content": "pregunta final"},
    ]
    if mensajes == esperado:
      print("\u2713 Orden correcto: system, ejemplos alternados user/assistant, pregunta final")
    else:
      print(f"\u2717 Se esperaba:\n    {esperado}\n  se obtuvo:\n    {mensajes}")
      fallas.append("orden")

  with patch.object(client.chat.completions, "create", return_value=_respuesta_falsa()) as mock_create:
    preguntar_con_estilo("pregunta final")
    if mock_create.call_args is None:
      print("\u2717 Nunca se llamo a client.chat.completions.create.")
      return
    mensajes = mock_create.call_args.kwargs.get("messages", [])
    if mensajes == [{"role": "user", "content": "pregunta final"}]:
      print("\u2713 Sin system_prompt ni ejemplos, solo va el mensaje final")
    else:
      print(f"\u2717 Se esperaba solo el mensaje final, se obtuvo: {mensajes}")
      fallas.append("sin extras")

  if not fallas:
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
  else:
    print(f"\n{len(fallas)} problema(s) por corregir.")

_validar_preguntar_con_estilo()


#### Tarea 2: Categorizar productos, con y sin ejemplos

Estas son las 6 categorias del catalogo y unos productos de prueba. Corre la celda y compara.


In [ ]:
CATEGORIAS = ["Topwear", "Bottomwear", "Shoes", "Bags", "Watches", "Innerwear"]

productos_prueba = [
    "Puma Men Grey T-shirt",
    "Fila Men Cush Flex Black Slippers",
    "Murcia Women Blue Handbag",
    "Levis Men Comfort Style Grey Innerwear Vest",
    "Manchester United Men Solid Black Track Pants",
]

instruccion = "Clasifica el producto en una de estas categorias: " + ", ".join(CATEGORIAS) + "."

print("=== ZERO-SHOT (sin ejemplos) ===")
for p in productos_prueba:
  print(f"{p:<48} -> {preguntar_con_estilo(f'{instruccion}\n\nProducto: {p}')}")


In [ ]:
ejemplos_catalogo = [
    ("Producto: Nike Women Purple Polo T-shirt", "Topwear"),
    ("Producto: Peter England Men Party Blue Jeans", "Bottomwear"),
    ("Producto: Titan Women Silver Watch", "Watches"),
]

print("=== FEW-SHOT (con 3 ejemplos) ===")
for p in productos_prueba:
  respuesta = preguntar_con_estilo(
      f"Producto: {p}",
      system_prompt=(
          "Eres un clasificador de productos de un e-commerce de ropa. "
          "Responde SOLO con una de estas categorias, sin explicaciones: "
          + ", ".join(CATEGORIAS) + "."
      ),
      ejemplos=ejemplos_catalogo,
  )
  print(f"{p:<48} -> {respuesta}")


Fijate en dos cosas. Primero, el **formato**: sin ejemplos el modelo suele explicarse ("Este producto es una camiseta, por lo tanto..."), y con ejemplos responde solo la etiqueta, que es lo que un sistema automatico necesita. Segundo, los casos dificiles: `Slippers` no contiene la palabra "shoes" y `Track Pants` podria confundirse con Topwear.

Todo esto salio gratis y en segundos. Guarda los resultados mentalmente: en la Actividad 3 los compararemos contra un modelo entrenado.


## Actividad 2: Via contexto

### Objetivo

El prompting no sirve cuando el modelo simplemente **no sabe** algo. Ningun LLM del mundo conoce el manual interno de la empresa de transporte.

La solucion mas simple: **pegarle el documento en el prompt**.

Primero creamos el manual como un archivo `.txt`, igual que si nos lo hubiera pasado el cliente.


#### Antes de empezar: sube el manual

El manual operativo de la empresa esta en el drive del curso, en el archivo `manual_transporte.txt`.

Descargalo y subelo a Colab: icono de la carpeta en el panel izquierdo, boton de subir archivo. Debe quedar en la raiz, al mismo nivel del notebook.

Luego ejecuta la celda para confirmar que quedo bien.


In [ ]:
import os

RUTA_MANUAL = "manual_transporte.txt"

if not os.path.exists(RUTA_MANUAL):
  print(f"\u2717 No encuentro '{RUTA_MANUAL}'.")
  print("    Subelo a Colab desde el panel de la izquierda (icono de carpeta).")
else:
  with open(RUTA_MANUAL, "r", encoding="utf-8") as f:
    manual = f.read()
  print(f"\u2713 Manual cargado: {len(manual)} caracteres, {len(manual.splitlines())} lineas\n")
  print(manual[:350] + "...")


#### Tarea 1: Preguntar SIN contexto

Antes de darle el manual, veamos que responde el modelo solo. Corre la celda.


In [ ]:
preguntas_manual = [
    "\u00bfCual es la carga maxima de fruto de palma por camion en Transportes del Llano?",
    "\u00bfCada cuantos kilometros se hace el mantenimiento preventivo?",
    "\u00bfCuanto se paga de viaticos en ruta nacional?",
]

for pregunta in preguntas_manual:
  print(f"P: {pregunta}")
  print(f"R: {preguntar_groq(pregunta)}\n")


El modelo no tiene forma de saberlo. Fijate si **admite** que no sabe o si **inventa** una cifra que suena plausible. Lo segundo es mucho mas peligroso: una respuesta inventada con formato correcto es dificil de detectar.


#### Tarea 2: Construir `responder_con_documento`

Ahora leemos el `.txt` completo y lo pegamos en el prompt antes de la pregunta.

### Requisitos

1. Abrir el archivo en `ruta_documento` y leer todo su contenido.
2. Armar un prompt que contenga el documento y la pregunta.
3. Llamar a `preguntar_groq` con un `system_prompt` que le exija responder **solo** con base en el documento, y decir que no esta si no aparece.


In [ ]:
def responder_con_documento(pregunta: str, ruta_documento: str = "manual_transporte.txt") -> str:
  """Lee un documento completo y le pide al LLM que responda solo con base en el."""

  # TODO: abrir el archivo en "ruta_documento" y leer todo su contenido en la variable "documento"
  #       Pista: with open(ruta_documento, "r", encoding="utf-8") as f:
  documento = ""

  # TODO: armar un prompt que contenga el documento y la pregunta

  # TODO: llamar a preguntar_groq(prompt, system_prompt=...) con un system prompt que le exija
  #       responder SOLO con base en el documento, y decir 'Eso no esta en el manual' si no aparece
  pass


**Evaluacion de implementacion**

In [ ]:
# @title
# Celda de validacion. No modificar.
from unittest.mock import patch

def _validar_responder_con_documento():
  if "responder_con_documento" not in globals():
    print("\u2717 Todavia no existe 'responder_con_documento'.")
    return

  ruta = "/tmp/_doc_de_prueba.txt"
  with open(ruta, "w", encoding="utf-8") as f:
    f.write("CONTENIDO DE PRUEBA: la clave es 4242.")

  fallas = []
  with patch("__main__.preguntar_groq", return_value="respuesta final") as mock_preg:
    try:
      obtenido = responder_con_documento("\u00bfcual es la clave?", ruta)
    except Exception as e:
      print(f"\u2717 responder_con_documento(...) lanzo {type(e).__name__}: {e}")
      return

    if not mock_preg.called:
      print("\u2717 No se esta llamando a 'preguntar_groq'")
      fallas.append("llamada")
    else:
      args, kwargs = mock_preg.call_args
      prompt_usado = args[0] if args else kwargs.get("prompt", "")

      if "CONTENIDO DE PRUEBA" in prompt_usado and "4242" in prompt_usado:
        print("\u2713 El prompt incluye el contenido del archivo")
      else:
        print("\u2717 El prompt no incluye el contenido del archivo. \u00bfLo estas leyendo con open()?")
        fallas.append("contenido")

      if "\u00bfcual es la clave?" in prompt_usado:
        print("\u2713 El prompt incluye la pregunta")
      else:
        print("\u2717 El prompt no incluye la pregunta")
        fallas.append("pregunta")

      system_usado = kwargs.get("system_prompt", "") or ""
      if "documento" in system_usado.lower():
        print("\u2713 El system_prompt le exige responder con base en el documento")
      else:
        print("\u2717 Falta un system_prompt que le diga que responda solo con base en el documento")
        fallas.append("system")

    if obtenido != "respuesta final":
      print(f"\u2717 Debia devolver lo que devuelve preguntar_groq, devolvio {obtenido!r}")
      fallas.append("retorno")
    else:
      print("\u2713 Devuelve el resultado de preguntar_groq")

  if not fallas:
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
  else:
    print(f"\n{len(fallas)} problema(s) por corregir.")

_validar_responder_con_documento()


#### Tarea 3: Las mismas preguntas, ahora con el manual

Agregamos ademas una pregunta que **no** esta en el manual.


In [ ]:
for pregunta in preguntas_manual:
  print(f"P: {pregunta}")
  print(f"R: {responder_con_documento(pregunta)}\n")

fuera = "\u00bfCuantos dias de vacaciones tienen los conductores?"
print(f"P: {fuera}")
print(f"R: {responder_con_documento(fuera)}")


Las tres primeras deberian salir exactas (12 toneladas, 10.000 km, 85.000 pesos) y la ultima deberia admitir que no esta en el manual.

**Lo importante: esto no escala.** Aqui el manual cabe entero en el prompt, pero si la empresa tuviera 500 manuales:

- No caben en la ventana de contexto.
- Pagarias por reenviar todos los documentos en **cada** pregunta.
- El modelo se distrae: con demasiado texto irrelevante, empeora.

La solucion es buscar primero los fragmentos relevantes y mandar solo esos. Eso es **RAG**, y es el tema de la proxima clase.


## Actividad 3: Via LoRA

### Objetivo

Volvemos a la tarea de la Actividad 1: **categorizar productos del catalogo**. Pero ahora, en vez de pedirselo a un modelo gigante por API, vamos a **entrenar** un modelo pequeno propio.

**LoRA** (*Low-Rank Adaptation*) congela el modelo base y entrena solo unas matrices pequenas anadidas a las capas de atencion. Se entrena una fraccion minima de los parametros.

El catalogo esta en el drive del curso, en `products_category.csv`: 4.800 productos reales de moda, ya balanceados (800 por categoria) y con la etiqueta numerica lista. **Subelo a Colab igual que el manual.**

> La instalacion fija versiones concretas: `peft` exige `torchao>=0.16` y Colab trae una mas vieja. Si Colab te pide reiniciar el entorno, reinicia y vuelve a ejecutar esta celda.


In [ ]:
!pip install -q torch "transformers>=4.44,<4.50" "torchao>=0.16.0" scikit-learn matplotlib seaborn pandas peft

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODELO_BASE = "bert-base-multilingual-cased"

print("Dispositivo:", device)


#### Tarea 1: Cargar el catalogo

El CSV trae tres columnas utiles: `productDisplayName` (el texto que vamos a clasificar), `subCategory` (el nombre de la categoria) y `label` (esa misma categoria como numero, que es lo que el modelo necesita).


In [ ]:
RUTA_CSV = "products_category.csv"

if not os.path.exists(RUTA_CSV):
  raise FileNotFoundError(
      f"No encuentro '{RUTA_CSV}'. Subelo a Colab desde el panel de la izquierda (icono de carpeta)."
  )

df = pd.read_csv(RUTA_CSV, index_col=0)

# Nombres de categoria ordenados por su id de etiqueta (0, 1, 2, ...)
id2etiqueta = df.drop_duplicates("label").set_index("label")["subCategory"].sort_index().to_dict()
NOMBRES = [id2etiqueta[i] for i in sorted(id2etiqueta)]

X_train, X_test, y_train, y_test = train_test_split(
    df["productDisplayName"].tolist(), df["label"].tolist(),
    test_size=0.2, random_state=42, stratify=df["label"].tolist(),
)

print(f"{len(X_train)} entrenamiento, {len(X_test)} prueba")
print(f"Categorias: {NOMBRES}\n")
print(df["subCategory"].value_counts(), "\n")
print("Ejemplos:")
for t, l in list(zip(X_train, y_train))[:5]:
  print(f"  {t:<50} -> {id2etiqueta[l]}")


#### Tarea 2: Configurar LoRA

- `r`: rango de las matrices que se entrenan (4 basico, 16 equilibrado, 64 potente).
- `lora_alpha`: intensidad de la adaptacion, tipicamente el doble de `r`.
- `lora_dropout`: regularizacion de los adaptadores.
- `target_modules`: que capas se adaptan. En BERT las proyecciones de atencion se llaman `query` y `value`. (En modelos tipo Llama o Qwen se llaman `q_proj` y `v_proj`: **el nombre depende de la arquitectura**.)
- `task_type`: aqui clasificacion de secuencias, `SEQ_CLS`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=len(NOMBRES)
).to(device)

config_lora = LoraConfig(
    # TODO: task_type=TaskType.SEQ_CLS
    # TODO: inference_mode=False
    # TODO: r=16
    # TODO: lora_alpha=32
    # TODO: lora_dropout=0.1
    # TODO: target_modules=["query", "value"]   (nombres de las capas en BERT)
)

# TODO: aplicar LoRA al modelo: modelo = get_peft_model(modelo, config_lora)

entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
total = sum(p.numel() for p in modelo.parameters())
print(f"Parametros entrenables: {entrenables:,} de {total:,} ({100 * entrenables / total:.2f}%)")


**Evaluacion de implementacion**

In [ ]:
# @title
# Celda de validacion. No modificar.

def _validar_config_lora():
  if "config_lora" not in globals() or config_lora is None:
    print("\u2717 Todavia no existe 'config_lora'.")
    return

  fallas = []

  for atributo, esperado in [("r", 16), ("lora_alpha", 32)]:
    obtenido = getattr(config_lora, atributo, None)
    if obtenido != esperado:
      print(f"\u2717 '{atributo}' debia ser {esperado}, es {obtenido}")
      fallas.append(atributo)
    else:
      print(f"\u2713 {atributo} = {esperado}")

  objetivos = set(getattr(config_lora, "target_modules", None) or [])
  if objetivos != {"query", "value"}:
    print(f"\u2717 'target_modules' debia ser {{'query', 'value'}} (nombres de BERT), es {objetivos}")
    if objetivos == {"q_proj", "v_proj"}:
      print("    Esos son los nombres de Llama/Qwen, no de BERT.")
    fallas.append("target_modules")
  else:
    print("\u2713 target_modules = {'query', 'value'}")

  if "SEQ_CLS" not in str(getattr(config_lora, "task_type", "")):
    print(f"\u2717 'task_type' debia ser SEQ_CLS, es {getattr(config_lora, 'task_type', None)}")
    fallas.append("task_type")
  else:
    print("\u2713 task_type = SEQ_CLS")

  if globals().get("modelo") is not None and hasattr(modelo, "parameters"):
    ent = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in modelo.parameters())
    pct = 100 * ent / tot if tot else 100
    if pct < 5:
      print(f"\u2713 Solo se entrena el {pct:.2f}% de los parametros ({ent:,} de {tot:,})")
    else:
      print(f"\u2717 Se esperaba entrenar menos del 5%, se entrena el {pct:.2f}%. \u00bfAplicaste get_peft_model?")
      fallas.append("porcentaje")

  if not fallas:
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
  else:
    print(f"\n{len(fallas)} problema(s) por corregir.")

_validar_config_lora()


#### Tarea 3: Medir el desempeno

`compute_metrics` recibe los **logits** y las etiquetas reales, y devuelve exactitud y F1.

La prediccion de cada ejemplo es la clase con el logit mas alto.


In [ ]:
def compute_metrics(eval_pred):
  """Calcula accuracy y F1 ponderado a partir de los logits."""

  logits, etiquetas = eval_pred

  # TODO: la prediccion de cada ejemplo es el indice del logit mas alto
  #       Pista: np.argmax(logits, axis=1)
  predicciones = None

  # TODO: calcular accuracy con accuracy_score(etiquetas, predicciones)
  # TODO: calcular precision, recall y f1 con
  #       precision_recall_fscore_support(etiquetas, predicciones, average="weighted")

  # TODO: devolver {"accuracy": ..., "f1": ..., "precision": ..., "recall": ...}
  pass


**Evaluacion de implementacion**

In [ ]:
# @title
# Celda de validacion. No modificar.
import numpy as np

def _validar_compute_metrics():
  if "compute_metrics" not in globals():
    print("\u2717 Todavia no existe 'compute_metrics'.")
    return

  logits = np.array([[0.1, 0.9], [0.8, 0.2], [0.3, 0.7]])
  etiquetas = np.array([1, 0, 0])
  try:
    obtenido = compute_metrics((logits, etiquetas))
  except Exception as e:
    print(f"\u2717 compute_metrics(...) lanzo {type(e).__name__}: {e}")
    return

  if obtenido is None:
    print("\u2717 Devolvio None. \u00bfTe falto el return?")
    return
  if not isinstance(obtenido, dict) or "accuracy" not in obtenido:
    print(f"\u2717 Debia devolver un dict con la clave 'accuracy', devolvio: {obtenido!r}")
    return

  if abs(obtenido["accuracy"] - 2 / 3) < 1e-6:
    print(f"\u2713 accuracy = {obtenido['accuracy']:.4f} (2 de 3 aciertos)")
  else:
    print(f"\u2717 Se esperaba accuracy=0.6667, se obtuvo {obtenido['accuracy']}")
    return

  if "f1" in obtenido:
    print(f"\u2713 Tambien devuelve f1 = {obtenido['f1']:.4f}")
    print("\n\U0001F389 \u00a1Todo funciona correctamente!")
  else:
    print("\u2717 Falta la clave 'f1' en el diccionario")

_validar_compute_metrics()


#### Tarea 4: Entrenar

Los titulos de producto son cortos, asi que `max_length=32` alcanza y hace el entrenamiento mucho mas rapido.

Fijate en el learning rate: `1e-4` es **diez veces mas alto** que en un fine-tuning completo. Los adaptadores LoRA son pocos parametros y necesitan una senal mas fuerte.


In [ ]:
train_enc = tokenizer(X_train, truncation=True, padding=True, max_length=32, return_tensors="pt")
test_enc = tokenizer(X_test, truncation=True, padding=True, max_length=32, return_tensors="pt")


class CatalogoDataset(torch.utils.data.Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self, idx):
    item = {k: v[idx] for k, v in self.encodings.items()}
    item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
    return item

  def __len__(self):
    return len(self.labels)


train_dataset = CatalogoDataset(train_enc, y_train)
test_dataset = CatalogoDataset(test_enc, y_test)

argumentos = TrainingArguments(
    output_dir="./modelo_catalogo_lora",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to=[],
)

entrenador = Trainer(
    model=modelo,
    args=argumentos,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

entrenador.train()


#### Tarea 5: Evaluar

In [ ]:
predicciones = entrenador.predict(test_dataset)
y_pred = np.argmax(predicciones.predictions, axis=1)

print(f"ACCURACY FINAL: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=NOMBRES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=NOMBRES, yticklabels=NOMBRES)
plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.title("Matriz de confusion")
plt.show()


Mira la matriz de confusion, no solo el accuracy. Las confusiones suelen tener sentido: `Shoes` con `Bags` casi nunca, pero `Topwear` con `Innerwear` si (una camiseta interior es las dos cosas segun el catalogo).


#### Tarea 6: Comparar con el prompting de la Actividad 1

Los mismos 5 productos, ahora clasificados por nuestro modelo entrenado.


In [ ]:
def categorizar(texto: str) -> str:
  entradas = tokenizer(texto, return_tensors="pt", truncation=True, padding=True, max_length=32)
  entradas = {k: v.to(device) for k, v in entradas.items()}
  with torch.no_grad():
    logits = modelo(**entradas).logits
  probabilidades = torch.softmax(logits, dim=-1)
  idx = int(torch.argmax(probabilidades, dim=-1))
  return f"{id2etiqueta[idx]} (confianza {probabilidades.max().item():.2f})"


for p in productos_prueba:
  print(f"{p:<48} -> {categorizar(p)}")


### Preguntas para discutir

**1. El modelo de la Actividad 1 tiene 20.000 millones de parametros; el de la Actividad 3 tiene 178 millones y solo entrenamos ~1% de ellos. Si los dos resuelven la tarea, ¿cual pondrias en produccion para categorizar 50.000 productos diarios?**

**2. En la Actividad 2 el manual completo cabia en el prompt. ¿Que dejaria de funcionar con 500 manuales, y que harias?**

**3. ¿Por que LoRA usa un learning rate diez veces mas alto que un fine-tuning completo?**

**4. `target_modules=["query", "value"]` funciona en BERT pero fallaria en Llama. ¿Por que, y como lo averiguarias para un modelo nuevo?**

### Extensiones opcionales

- **Menos texto**: recorta cada titulo a sus 3 primeras palabras y reentrena. ¿Cuanta informacion estaba en el resto?
- **Menos datos**: baja `N_POR_CLASE` a 100 y reentrena. ¿Cuantos ejemplos por clase hacen falta para superar al prompting?
- **Rango de LoRA**: prueba `r=4` y `r=64`. ¿Mas capacidad siempre es mejor?
- **Sin LoRA**: entrena el mismo BERT sin `get_peft_model` y compara tiempo, memoria y accuracy.
- **Etiquetar con el grande**: usa `preguntar_con_estilo` para etiquetar 200 productos y entrena solo con esas etiquetas. ¿Que tan lejos queda del modelo entrenado con las etiquetas reales?
